In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

# Cấu hình thiết bị
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# MODEL CONFIG
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LEN = 256
BATCH_SIZE = 32
LEARNING_RATE = 2e-5

# Sentiment Mapping (0-3)
ID2SENTIMENT = {
    0: "Neutral",
    1: "Positive",
    2: "Negative",
    3: "Toxic"
}
SENTIMENT2ID = {v: k for k, v in ID2SENTIMENT.items()}

# Topic Mapping (0-9)
ID2TOPIC = {
    0: "Spam",
    1: "News",
    2: "Academic",
    3: "Other",
    4: "Service",
    5: "Jobs & Recruitment",
    6: "Personal Affairs",
    7: "Social Affairs",
    8: "Help & Share",
    9: "Club & Events"
}
TOPIC2ID = {v: k for k, v in ID2TOPIC.items()}

Using device: cuda


In [2]:
class NEUMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['text'].values
        # Đảm bảo dữ liệu label là số nguyên
        self.sentiments = df['sentiment'].values
        self.topics = df['topic'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = str(self.texts[index])
        sentiment = self.sentiments[index]
        topic = self.topics[index]

        # Tokenize (quan trọng cho PhoBERT)
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(sentiment, dtype=torch.long),
            'topic_label': torch.tensor(topic, dtype=torch.long)
        }

In [3]:
class MultiTaskPhoBERT(nn.Module):
    def __init__(self, n_sentiment_classes, n_topic_classes):
        super(MultiTaskPhoBERT, self).__init__()
        # Load PhoBERT base v2
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        
        # Head 1: Sentiment Classification
        self.sentiment_out = nn.Linear(self.bert.config.hidden_size, n_sentiment_classes)
        
        # Head 2: Topic Classification
        self.topic_out = nn.Linear(self.bert.config.hidden_size, n_topic_classes)

    def forward(self, input_ids, attention_mask):
        # Lấy output từ PhoBERT
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Lấy vector đại diện cho cả câu (pooled_output thường là token [CLS])
        pooled_output = outputs.pooler_output 
        output = self.drop(pooled_output)
        
        # Đưa qua 2 nhánh riêng biệt
        sent_logits = self.sentiment_out(output)
        topic_logits = self.topic_out(output)
        
        return sent_logits, topic_logits

In [ ]:
def get_class_weights(series):
    # Sử dụng sklearn để tính trọng số nghịch đảo
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(series),
        y=series
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)

try:
    df_train = pd.read_csv("/kaggle/input/neu-esc-dataset/train_set.csv")
    df_val = pd.read_csv("/kaggle/input/neu-esc-dataset/val_set.csv")
    
    # Chuẩn hóa tên cột nếu cần (dựa trên bài trước cột topic tên là 'classification')
    if 'classification' in df_train.columns:
        df_train = df_train.rename(columns={'classification': 'topic'})
        df_val = df_val.rename(columns={'classification': 'topic'})

    # TÍNH TRỌNG SỐ LOSS
    sentiment_weights = get_class_weights(df_train['sentiment'])
    topic_weights = get_class_weights(df_train['topic'])

    print("Sentiment Weights:", sentiment_weights)
    print("Topic Weights:", topic_weights)

except Exception as e:
    print(f"Chưa load được file csv thực tế để tính weight. Lỗi: {e}")
    # Nếu test chạy thử không có file, dùng weight giả lập bằng 1
    sentiment_weights = torch.ones(len(ID2SENTIMENT)).to(device)
    topic_weights = torch.ones(len(ID2TOPIC)).to(device)

Sentiment Weights: tensor([0.3616, 1.9890, 1.5873, 9.8496], device='cuda:0')
Topic Weights: tensor([8.2021, 3.6701, 0.3134, 0.2287, 1.3994, 4.0865, 2.2333, 4.3000, 4.9566,
        4.9887], device='cuda:0')


In [ ]:
# 1. Khởi tạo Tokenizer & DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NEUMultiTaskDataset(df_train, tokenizer, MAX_LEN)
val_dataset = NEUMultiTaskDataset(df_val, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# 2. Khởi tạo Model
model = MultiTaskPhoBERT(
    n_sentiment_classes=len(ID2SENTIMENT),
    n_topic_classes=len(ID2TOPIC)
)
model = model.to(device)

# 3. Loss Function & Optimizer
# Truyền class_weights vào CrossEntropyLoss để phạt nặng model nếu đoán sai lớp ít mẫu
criterion_sent = nn.CrossEntropyLoss(weight=sentiment_weights)
criterion_topic = nn.CrossEntropyLoss(weight=topic_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

from tqdm.auto import tqdm # Thư viện tạo thanh loading đẹp mắt

def train_epoch(model, data_loader, loss_fn_sent, loss_fn_topic, optimizer, device):
    model = model.train()
    total_loss = 0
    
    # Bọc data_loader vào tqdm để hiện thanh tiến trình
    # desc: Mô tả hiện ở đầu thanh
    progress_bar = tqdm(data_loader, desc="Training", position=0, leave=True)
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sent_labels = batch['sentiment_label'].to(device)
        topic_labels = batch['topic_label'].to(device)

        # Forward
        sent_logits, topic_logits = model(input_ids, attention_mask)

        # Tính loss
        loss_s = loss_fn_sent(sent_logits, sent_labels)
        loss_t = loss_fn_topic(topic_logits, topic_labels)
        loss = loss_s + loss_t 

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        progress_bar.set_postfix({'loss': loss.item()})
        
    return total_loss / len(data_loader)

print("Bắt đầu train...")


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-01-11 19:12:55.508780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768158775.706165      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768158775.759295      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768158776.206173      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768158776.206212      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768158776.206215      24 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Bắt đầu train...


In [ ]:
NUM_EPOCHS = 20 

history = {
    'train_loss': [],
    'train_acc': [] 
}

print(f"Bắt đầu huấn luyện {NUM_EPOCHS} epochs trên thiết bị: {device}")
print("="*30)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 10)
    
    train_loss = train_epoch(
        model, 
        train_loader, 
        criterion_sent, 
        criterion_topic, 
        optimizer, 
        device
    )
    

    history['train_loss'].append(train_loss)
    

    print(f" -> Train Loss: {train_loss:.4f}")
    

print("\nHOÀN THÀNH HUẤN LUYỆN!")

Bắt đầu huấn luyện 20 epochs trên thiết bị: cuda

Epoch 1/20
----------


Training:   0%|          | 0/721 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

 -> Train Loss: 2.3952

Epoch 2/20
----------


Training:   0%|          | 0/721 [00:00<?, ?it/s]

...

 -> Train Loss: 0.1657

Epoch 19/20
----------


Training:   0%|          | 0/721 [00:00<?, ?it/s]

 -> Train Loss: 0.1192

Epoch 20/20
----------


Training:   0%|          | 0/721 [00:00<?, ?it/s]

 -> Train Loss: 0.1016

HOÀN THÀNH HUẤN LUYỆN!


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import torch
from tqdm.auto import tqdm

print("Đang chuẩn bị dữ liệu Test...")

# Load file test
df_test = pd.read_csv("/kaggle/input/neu-esc-dataset/test_set.csv")

if 'classification' in df_test.columns:
    df_test = df_test.rename(columns={'classification': 'topic'})

# Tạo Dataset và DataLoader
test_dataset = NEUMultiTaskDataset(df_test, tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


def evaluate_model(model, data_loader, device):
    model = model.eval() 
    true_sentiments = []
    pred_sentiments = []
    
    true_topics = []
    pred_topics = []
    
    print("Đang chạy dự đoán trên tập Test...")
    with torch.no_grad(): # Tắt tính toán gradient để chạy nhanh hơn
        for batch in tqdm(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            sent_labels = batch['sentiment_label'].cpu().numpy()
            topic_labels = batch['topic_label'].cpu().numpy()
            
            # Dự đoán (Forward)
            sent_logits, topic_logits = model(input_ids, attention_mask)
            
            # Lấy nhãn có xác suất cao nhất (Argmax)
            sent_preds = torch.argmax(sent_logits, dim=1).cpu().numpy()
            topic_preds = torch.argmax(topic_logits, dim=1).cpu().numpy()
            
            # Lưu lại
            true_sentiments.extend(sent_labels)
            pred_sentiments.extend(sent_preds)
            
            true_topics.extend(topic_labels)
            pred_topics.extend(topic_preds)
            
    return true_sentiments, pred_sentiments, true_topics, pred_topics


y_true_sent, y_pred_sent, y_true_topic, y_pred_topic = evaluate_model(model, test_loader, device)

# --- BÁO CÁO CẢM XÚC (SENTIMENT) ---
print("\n" + "="*50)
print("KẾT QUẢ ĐÁNH GIÁ: CẢM XÚC (SENTIMENT)")
print("="*50)
print(f"Độ chính xác (Accuracy): {accuracy_score(y_true_sent, y_pred_sent):.4f}\n")

sent_target_names = [ID2SENTIMENT[i] for i in range(len(ID2SENTIMENT))]

print(classification_report(y_true_sent, y_pred_sent, target_names=sent_target_names))


# --- BÁO CÁO CHỦ ĐỀ (TOPIC) ---
print("\n" + "="*50)
print("KẾT QUẢ ĐÁNH GIÁ: CHỦ ĐỀ (TOPIC)")
print("="*50)
print(f"Độ chính xác (Accuracy): {accuracy_score(y_true_topic, y_pred_topic):.4f}\n")

topic_target_names = [ID2TOPIC[i] for i in range(len(ID2TOPIC))]
print(classification_report(y_true_topic, y_pred_topic, target_names=topic_target_names))

Đang chuẩn bị dữ liệu Test...
Đang chạy dự đoán trên tập Test...


  0%|          | 0/207 [00:00<?, ?it/s]


KẾT QUẢ ĐÁNH GIÁ: CẢM XÚC (SENTIMENT)
Độ chính xác (Accuracy): 0.8216

              precision    recall  f1-score   support

     Neutral       0.88      0.87      0.88      4558
    Positive       0.68      0.69      0.68       834
    Negative       0.67      0.70      0.68      1046
       Toxic       0.88      0.86      0.87       175

    accuracy                           0.82      6613
   macro avg       0.78      0.78      0.78      6613
weighted avg       0.82      0.82      0.82      6613


KẾT QUẢ ĐÁNH GIÁ: CHỦ ĐỀ (TOPIC)
Độ chính xác (Accuracy): 0.7661

                    precision    recall  f1-score   support

              Spam       0.50      0.62      0.56        84
              News       0.42      0.56      0.48       183
          Academic       0.84      0.82      0.83      2104
             Other       0.86      0.83      0.84      2882
           Service       0.62      0.66      0.64       474
Jobs & Recruitment       0.58      0.67      0.62       163
  Per

In [8]:
def predict_sentence(text, model, tokenizer, device):
    model.eval()
    
    # Tokenize input
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_token_type_ids=False,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        # Lấy raw output (logits)
        sent_logits, topic_logits = model(input_ids, attention_mask)
        
        # Chọn class có xác suất cao nhất (Argmax)
        _, sent_pred_id = torch.max(sent_logits, dim=1)
        _, topic_pred_id = torch.max(topic_logits, dim=1)
        
        # Map từ ID ngược lại ra Text Label
        sent_pred_label = ID2SENTIMENT[sent_pred_id.item()]
        topic_pred_label = ID2TOPIC[topic_pred_id.item()]
        
    return [sent_pred_label, topic_pred_label]

# --- CHẠY THỬ NGHIỆM ---
test_sentence_1 = "Trường này học phí đắt quá mà cơ sở vật chất chán đời."
test_sentence_2 = "Sắp tới có sự kiện chào tân sinh viên vui lắm mọi người ơi."
test_sentence_3 = "Vãi cả đái sao thằng ngu như mày lại có thể chung lớp với tao được."

# Giả sử model đã được train, ta gọi hàm:
print(f"Câu 1: {test_sentence_1}")
print(f"Output: {predict_sentence(test_sentence_1, model, tokenizer, device)}")

print(f"\nCâu 2: {test_sentence_2}")
print(f"Output: {predict_sentence(test_sentence_2, model, tokenizer, device)}")

print(f"\nCâu 3: {test_sentence_3}")
print(f"Output: {predict_sentence(test_sentence_3, model, tokenizer, device)}")

Câu 1: Trường này học phí đắt quá mà cơ sở vật chất chán đời.
Output: ['Negative', 'Service']

Câu 2: Sắp tới có sự kiện chào tân sinh viên vui lắm mọi người ơi.
Output: ['Positive', 'Club & Events']

Câu 3: Vãi cả đái sao thằng ngu như mày lại có thể chung lớp với tao được.
Output: ['Toxic', 'Personal Affairs']


In [ ]:
import os
import json
import torch

def save_pipeline(model, tokenizer, save_dir):
    """
    Hàm lưu toàn bộ thành phần cần thiết để deploy/inference sau này.
    """
    # 1. Tạo thư mục lưu trữ
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Đã tạo thư mục: {save_dir}")

    # 2. Lưu weights của Model (chỉ lưu state_dict cho nhẹ)
    model_path = os.path.join(save_dir, "phobert_multitask_weights.pth")
    torch.save(model.state_dict(), model_path)
    print(f"✅ Đã lưu Model weights tại: {model_path}")

    # 3. Lưu Tokenizer (quan trọng để bộ tách từ khớp với lúc train)
    tokenizer.save_pretrained(save_dir)
    print(f"✅ Đã lưu Tokenizer tại: {save_dir}")

    # 4. Lưu Config & Mapping (ID -> Label)
    config_data = {
        "base_model": MODEL_NAME,
        "max_len": MAX_LEN,
        "id2sentiment": ID2SENTIMENT,
        "sent2id": SENTIMENT2ID,
        "id2topic": ID2TOPIC,
        "topic2id": TOPIC2ID
    }
    
    config_path = os.path.join(save_dir, "config.json")
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, ensure_ascii=False, indent=4)
    print(f"✅ Đã lưu Config & Mapping tại: {config_path}")
    print("\nLưu thành công toàn bộ pipeline!")

# --- THỰC HIỆN LƯU ---
SAVE_DIRECTORY = "phobert_multitask_v2" # Tên folder bạn muốn lưu
save_pipeline(model, tokenizer, SAVE_DIRECTORY)

Đã tạo thư mục: phobert_multitask_v2
✅ Đã lưu Model weights tại: phobert_multitask_v2/phobert_multitask_weights.pth
✅ Đã lưu Tokenizer tại: phobert_multitask_v2
✅ Đã lưu Config & Mapping tại: phobert_multitask_v2/config.json

Lưu thành công toàn bộ pipeline!
